# Unified FR/NFR Classification and FR→NFR Trace Link Discovery — extended experiments (v2)

**What this notebook does**
* **D1** – your e-commerce corpus (LLM-generated, expert-reviewed). You upload `Labeled_Requirements_Dataset.csv`.
* **D2** – PROMISE_exp (969 requirements, 15 projects), a public benchmark. Downloaded automatically.
* **D3** – PROMISE NFR manual FR→NFR trace matrices (10 projects), giving *real ground truth* for linking. Downloaded automatically.

Experiments: held-out evaluation with McNemar tests, repeated 5×2 cross-validation with Alpaydin's F-test,
Wilcoxon tests and effect sizes, leave-one-project-out (cross-project) evaluation, cross-dataset transfer
(synthetic → real and back), ablations, calibration, lexical vs. semantic vs. hybrid linking, and efficiency.

**Run:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all. Upload the CSV when asked.
Total time is roughly 45–90 minutes with the GPU (set `RUN_BERT = False` for a ~15-minute run without BERT).
At the end `results_v2_bundle.zip` downloads: all tables (CSV), figures (300-dpi PNG) and `results_v2.json`.

In [ ]:
# ---------- 0. Setup ----------
!pip -q install imbalanced-learn==0.12.4 sentence-transformers rank_bm25 psutil

In [ ]:
import os, re, json, time, random, platform, warnings, zipfile, glob, subprocess, itertools
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import psutil, sklearn, imblearn, scipy
from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RepeatedStratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score, confusion_matrix,
                             average_precision_score, brier_score_loss)
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
warnings.filterwarnings("ignore")

# ---------------- configuration ----------------
SEED = 42
RUN_NEURAL = True          # SBERT embeddings (classification + linking)
RUN_BERT = True            # fine-tuned BERT classifier (needs GPU for reasonable speed)
BERT_IN_CV = True          # include BERT in the 5x2 CV (FR/NFR task only); adds ~20 fine-tuning runs
SBERT_NAME = "sentence-transformers/all-MiniLM-L6-v2"
BERT_NAME = "bert-base-uncased"
BERT_EPOCHS, BERT_LR, BERT_BS, BERT_MAXLEN = 4, 2e-5, 32, 64
PROMISE_REPO = "https://github.com/AleksandarMitrevski/se-requirements-classification"
PROMISE_COMMIT = "1f5dc4501a1956f21011083594d925bac49f198c"   # pinned for reproducibility
OUT = "results_v2"; os.makedirs(OUT, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except Exception: pass
set_seed()
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9})
R = {"seed": SEED, "env": {"python": platform.python_version(), "sklearn": sklearn.__version__, "imblearn": imblearn.__version__,
                           "numpy": np.__version__, "scipy": scipy.__version__}}
def save_fig(name): plt.tight_layout(); plt.savefig(f"{OUT}/{name}.png", bbox_inches="tight"); plt.show(); plt.close()
def save_tab(df, name): df.to_csv(f"{OUT}/{name}.csv", index=False); print(f"\n== {name} ==\n", df.to_string(index=False))

In [ ]:
# ---------- 1. Preprocessing ----------
USE_LEMMATIZER = True
try:
    import nltk
    nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)
    from nltk.stem import WordNetLemmatizer
    _lem = WordNetLemmatizer(); _lem.lemmatize("tests")
except Exception as e:
    print("WordNet unavailable -> lemmatisation disabled:", e); USE_LEMMATIZER = False
R["lemmatization_available"] = USE_LEMMATIZER
STOP = set("""i me my myself we our ours ourselves you you're you've you'll you'd your yours yourself yourselves he him his himself she she's her hers herself it it's its itself they them their theirs themselves what which who whom this that that'll these those am is are was were be been being have has had having do does did doing a an the and but if or because as until while of at by for with about against between into through during before after above below to from up down in out on off over under again further then once here there when where why how all any both each few more most other some such no nor not only own same so than too very s t can will just don don't should should've now d ll m o re ve y ain aren aren't couldn couldn't didn didn't doesn doesn't hadn hadn't hasn hasn't haven haven't isn isn't ma mightn mightn't mustn mustn't needn needn't shan shan't shouldn shouldn't wasn wasn't weren weren't won won't wouldn wouldn't""".split())
TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z0-9]+")
def preprocess(text, lemmatize=True, stopwords=True):
    toks = TOKEN_RE.findall(str(text))
    if stopwords: toks = [t for t in toks if t.lower() not in STOP]
    toks = [t for t in toks if len(t) > 1]
    if lemmatize and USE_LEMMATIZER: toks = [_lem.lemmatize(_lem.lemmatize(t.lower(), "n"), "v") for t in toks]
    return " ".join(t.lower() for t in toks)

def jaccard_matrix(a, b):
    cv = CountVectorizer(binary=True, token_pattern=r"\S+").fit(list(a) + list(b))
    A = cv.transform(a).astype(np.float32); B = cv.transform(b).astype(np.float32)
    inter = (A @ B.T).toarray(); sa = np.asarray(A.sum(1)).ravel()[:, None]; sb = np.asarray(B.sum(1)).ravel()[None, :]
    return inter / np.maximum(sa + sb - inter, 1)

def clean_corpus(df, name, min_sub=10):
    """df columns: text, type (FR/NFR), sub, (module), (project). Removes empties, exact and near duplicates, rare subtypes."""
    n0 = len(df); df = df.copy()
    df["clean"] = df["text"].map(preprocess)
    df["clean_nolem"] = df["text"].map(lambda t: preprocess(t, lemmatize=False))
    df["clean_nostop"] = df["text"].map(lambda t: preprocess(t, stopwords=False))
    df = df[df["clean"].str.len() > 0]
    df["_norm"] = df["text"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip(" .")
    df = df.drop_duplicates("_norm").reset_index(drop=True); n_exact = n0 - len(df)
    J = jaccard_matrix(df["clean"], df["clean"]); np.fill_diagonal(J, 0); drop = set()
    for i in range(len(df)):
        if i in drop: continue
        drop.update(j for j in np.where(J[i] >= 0.95)[0] if j > i)
    df = df.drop(index=sorted(drop)).reset_index(drop=True)
    sc = df[df.type == "NFR"]["sub"].value_counts(); rare = sc[sc < min_sub].index.tolist()
    n_rare = int(df["sub"].isin(rare).sum()); df = df[~df["sub"].isin(rare)].reset_index(drop=True)
    info = {"raw": n0, "exact_dups": n_exact, "near_dups": len(drop), "rare_subtypes_removed": rare, "rare_removed": n_rare,
            "final": len(df), "type_counts": df.type.value_counts().to_dict(), "sub_counts": df[df.type == "NFR"]["sub"].value_counts().to_dict()}
    print(name, json.dumps(info)); return df, info

In [ ]:
# ---------- 2. Data ----------
# D1: user corpus
CSV = "Labeled_Requirements_Dataset.csv"
if not os.path.exists(CSV):
    from google.colab import files
    up = files.upload(); CSV = list(up.keys())[0]
raw1 = pd.read_csv(CSV); raw1.columns = [c.strip() for c in raw1.columns]
d1 = pd.DataFrame({"text": raw1["Requirement Text"].astype(str).str.strip(), "type": raw1["Type"].astype(str).str.strip(),
                   "sub": raw1["Subcategory"].astype(str).str.strip(),
                   "module": raw1["Source/Module"].astype(object).where(raw1["Source/Module"].notna(), "Unknown").astype(str).str.strip()})
d1, R["D1_info"] = clean_corpus(d1, "D1"); R["D1_info"]["n_modules"] = int(d1.module.nunique())

# D2 + D3: PROMISE_exp and PROMISE NFR traces (pinned commit)
if not os.path.exists("promise_repo"):
    subprocess.run(["git", "clone", "-q", PROMISE_REPO, "promise_repo"], check=True)
    subprocess.run(["git", "-C", "promise_repo", "checkout", "-q", PROMISE_COMMIT], check=True)
CODES = {"F": "Functional", "A": "Availability", "FT": "Fault Tolerance", "L": "Legal", "LF": "Look & Feel", "MN": "Maintainability",
         "O": "Operability", "PE": "Performance", "PO": "Portability", "SC": "Scalability", "SE": "Security", "US": "Usability"}
raw2 = pd.read_csv("promise_repo/1-exploratory-analysis/data/PROMISE_exp.csv")
d2 = pd.DataFrame({"text": raw2["RequirementText"].astype(str).str.strip(), "type": np.where(raw2["_class_"] == "F", "FR", "NFR"),
                   "sub": raw2["_class_"].map(CODES), "project": raw2["ProjectID"].astype(int)})
d2, R["D2_info"] = clean_corpus(d2, "D2")

def read_arff_rows(path):
    rows, data = [], False
    for l in open(path, encoding="latin-1"):
        l = l.strip()
        if l.lower().startswith("@data"): data = True; continue
        if data and l and not l.startswith("%"): rows.append(l)
    return rows
base = "promise_repo/0-datasets/tera-PROMISE NFR"
nfr_rows = []
for l in read_arff_rows(f"{base}/nfr.arff"):
    m = re.match(r"^(\d+),'(.*)',(\w+)$", l)
    if m: nfr_rows.append((int(m[1]), m[2].replace("\\'", "'"), m[3]))
promise_nfr = pd.DataFrame(nfr_rows, columns=["project", "text", "label"])
D3 = {}
for p in range(1, 11):
    proj = promise_nfr[promise_nfr.project == p].reset_index(drop=True)          # R1..Rn = order within project
    lines = read_arff_rows(f"{base}/P(1-10)_NFR_FR_trace/P{p}_NFR_FR_trace.arff")
    hdr = open(f"{base}/P(1-10)_NFR_FR_trace/P{p}_NFR_FR_trace.arff", encoding="latin-1").read()
    n_nfr = len(re.findall(r"@attribute\s+NFR\d+", hdr))
    nfr_idx = [i for i in range(len(proj)) if proj.label[i] != "F"][:n_nfr]
    fr_ids, G = [], []
    for l in lines:
        parts = [x.strip() for x in l.split(",")]
        fr_ids.append(int(parts[0][1:]) - 1); G.append([int(x) for x in parts[1:1 + n_nfr]])
    D3[p] = {"fr": proj.text[fr_ids].tolist(), "nfr": proj.text[nfr_idx].tolist(), "G": np.array(G, dtype=bool),
             "fr_types_ok": bool((proj.label[fr_ids] == "F").all())}
R["D3_info"] = {p: {"n_fr": len(v["fr"]), "n_nfr": len(v["nfr"]), "links": int(v["G"].sum()), "density": float(v["G"].mean()),
                    "fr_types_ok": v["fr_types_ok"]} for p, v in D3.items()}
print(pd.DataFrame(R["D3_info"]).T)

In [ ]:
# ---------- 3. Representations and models ----------
def make_vec(ngram=(1, 2)): return TfidfVectorizer(ngram_range=ngram, min_df=2, sublinear_tf=True, norm="l2", token_pattern=r"\S+")
# note: min_df=2 is used for all datasets so the smaller PROMISE_exp keeps enough vocabulary
SBERT = None; EMB = {}
if RUN_NEURAL:
    from sentence_transformers import SentenceTransformer
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"; print("Device:", DEVICE)
    SBERT = SentenceTransformer(SBERT_NAME, device=DEVICE)
    R["env"]["device"] = DEVICE
def embed(texts):
    key = tuple(texts)
    if key not in EMB: EMB[key] = SBERT.encode(list(texts), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    return EMB[key]

cv3 = lambda: StratifiedKFold(3, shuffle=True, random_state=SEED)
class TfidfModel:
    """TF-IDF features + a classical learner. text_col chooses the preprocessing variant (ablations)."""
    def __init__(self, kind, text_col="clean", ngram=(1, 2), oversample=True, calibrate="isotonic"):
        self.kind, self.text_col, self.ngram, self.oversample, self.calibrate = kind, text_col, ngram, oversample, calibrate
    def fit(self, df, y):
        self.vec = make_vec(self.ngram).fit(df[self.text_col]); X = self.vec.transform(df[self.text_col])
        steps = [("ros", RandomOverSampler(random_state=SEED))] if self.oversample else []
        if self.kind == "linsvc": est, grid = LinearSVC(max_iter=20000, random_state=SEED), {"m__C": [0.03, 0.1, 0.3, 1, 3]}
        elif self.kind == "lr": est, grid = LogisticRegression(max_iter=5000, class_weight="balanced"), {"m__C": [1, 10, 100]}
        elif self.kind == "rbf": est, grid = SVC(kernel="rbf", random_state=SEED), {"m__C": [1, 10]}
        elif self.kind == "rf": est, grid = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1), {"m__max_features": ["sqrt"]}
        elif self.kind == "gb": est, grid = GradientBoostingClassifier(n_estimators=150, random_state=SEED), {"m__max_depth": [3]}
        elif self.kind == "nb": est, grid = ComplementNB(), {"m__alpha": [0.1, 0.5, 1.0]}
        k = max(2, min(3, pd.Series(y).value_counts().min()))
        gs = GridSearchCV(ImbPipeline(steps + [("m", est)]), grid, scoring="f1_macro",
                          cv=StratifiedKFold(k, shuffle=True, random_state=SEED), n_jobs=-1).fit(X, y)
        self.model = gs.best_estimator_; self.best = gs.best_params_
        self.binary = len(set(y)) == 2
        if self.kind == "linsvc" and self.binary and self.calibrate:
            Xr, yr = (RandomOverSampler(random_state=SEED).fit_resample(X, y) if self.oversample else (X, y))
            self.cal = CalibratedClassifierCV(LinearSVC(C=self.best["m__C"], max_iter=20000, random_state=SEED),
                                              method=self.calibrate, cv=3).fit(Xr, yr)
        else: self.cal = None
        return self
    def predict(self, df):
        X = self.vec.transform(df[self.text_col])
        return self.cal.predict(X) if self.cal is not None else self.model.predict(X)
    def proba(self, df):
        X = self.vec.transform(df[self.text_col])
        return self.cal.predict_proba(X)[:, 1] if self.cal is not None else None

class SbertLR:
    def fit(self, df, y):
        X = embed(df["text"].tolist())
        self.m = GridSearchCV(LogisticRegression(max_iter=5000, class_weight="balanced"), {"C": [0.3, 1, 3, 10]}, scoring="f1_macro",
                              cv=StratifiedKFold(max(2, min(3, pd.Series(y).value_counts().min())), shuffle=True, random_state=SEED), n_jobs=-1).fit(X, y)
        return self
    def predict(self, df): return self.m.predict(embed(df["text"].tolist()))
    def proba(self, df): return self.m.predict_proba(embed(df["text"].tolist()))[:, 1] if len(self.m.classes_) == 2 else None

class BertClf:
    def fit(self, df, y):
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        set_seed()
        self.labels = sorted(set(y)); lid = {l: i for i, l in enumerate(self.labels)}; yy = np.array([lid[v] for v in y])
        self.tok = AutoTokenizer.from_pretrained(BERT_NAME)
        self.model = AutoModelForSequenceClassification.from_pretrained(BERT_NAME, num_labels=len(self.labels), ignore_mismatched_sizes=True).to(DEVICE)
        enc = self.tok(df["text"].tolist(), truncation=True, padding="max_length", max_length=BERT_MAXLEN, return_tensors="pt")
        ds = torch.utils.data.TensorDataset(enc["input_ids"], enc["attention_mask"], torch.tensor(yy))
        dl = torch.utils.data.DataLoader(ds, batch_size=BERT_BS, shuffle=True, generator=torch.Generator().manual_seed(SEED))
        w = torch.tensor(len(yy) / (len(self.labels) * np.bincount(yy, minlength=len(self.labels)).clip(1)), dtype=torch.float).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=BERT_LR, weight_decay=0.01)
        steps = BERT_EPOCHS * len(dl); sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / max(1, int(0.1 * steps))) * max(0.0, (steps - s) / steps))
        lossf = torch.nn.CrossEntropyLoss(weight=w); self.model.train()
        for _ in range(BERT_EPOCHS):
            for ids, am, lab in dl:
                opt.zero_grad(); out = self.model(input_ids=ids.to(DEVICE), attention_mask=am.to(DEVICE))
                loss = lossf(out.logits, lab.to(DEVICE)); loss.backward(); torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0); opt.step(); sched.step()
        return self
    def _logits(self, df):
        import torch
        self.model.eval(); outs = []
        texts = df["text"].tolist()
        with torch.no_grad():
            for i in range(0, len(texts), 128):
                enc = self.tok(texts[i:i + 128], truncation=True, padding=True, max_length=BERT_MAXLEN, return_tensors="pt").to(DEVICE)
                outs.append(self.model(**enc).logits.float().cpu().numpy())
        return np.vstack(outs)
    def predict(self, df): return np.array(self.labels)[self._logits(df).argmax(1)]
    def proba(self, df):
        if len(self.labels) != 2: return None
        z = self._logits(df); e = np.exp(z - z.max(1, keepdims=True)); return (e / e.sum(1, keepdims=True))[:, self.labels.index("NFR")]

def model_zoo(task, for_cv=False):
    zoo = {"TF-IDF + Cal. LinearSVC (proposed)": lambda: TfidfModel("linsvc"),
           "TF-IDF + Logistic Regression": lambda: TfidfModel("lr"),
           "TF-IDF + Complement NB": lambda: TfidfModel("nb"),
           "TF-IDF + RBF-SVM": lambda: TfidfModel("rbf"),
           "TF-IDF + Random Forest": lambda: TfidfModel("rf")}
    if task == "binary": zoo["TF-IDF + Gradient Boosting"] = lambda: TfidfModel("gb")
    if RUN_NEURAL: zoo["SBERT + Logistic Regression"] = lambda: SbertLR()
    if RUN_BERT and RUN_NEURAL and (not for_cv or (BERT_IN_CV and task == "binary")): zoo["Fine-tuned BERT"] = lambda: BertClf()
    return zoo
PROPOSED = "TF-IDF + Cal. LinearSVC (proposed)"

def task_data(df, task):
    if task == "binary": return df, df["type"].values
    sub = df[df.type == "NFR"].reset_index(drop=True); return sub, sub["sub"].values

In [ ]:
# ---------- 4. Statistics helpers ----------
def boot_ci(fn, n, B=1000, seed=SEED):
    rng = np.random.default_rng(seed); v = []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        try: v.append(fn(idx))
        except Exception: pass
    return [round(float(np.percentile(v, 2.5)), 4), round(float(np.percentile(v, 97.5)), 4)]
def mcnemar(y, a, b):
    ca, cb = (a == y), (b == y); n01 = int((ca & ~cb).sum()); n10 = int((~ca & cb).sum())
    if n01 + n10 == 0: return 1.0
    return float(stats.binomtest(min(n01, n10), n01 + n10, 0.5).pvalue)      # exact McNemar
def a12(x, y):
    x, y = np.asarray(x), np.asarray(y); return float(((x[:, None] > y[None, :]).sum() + 0.5 * (x[:, None] == y[None, :]).sum()) / (len(x) * len(y)))
def alpaydin_f(diffs):  # diffs: shape (5,2) of score differences from 5x2 CV
    d = np.asarray(diffs); s2 = ((d - d.mean(1, keepdims=True)) ** 2).sum(1)
    if s2.sum() == 0: return 1.0
    F = (d ** 2).sum() / (2 * s2.sum()); return float(1 - stats.f.cdf(F, 10, 5))
def holm(pvals):
    p = np.asarray(pvals, float); o = np.argsort(p); adj = np.empty_like(p); m = len(p); run = 0
    for r, i in enumerate(o): run = max(run, (m - r) * p[i]); adj[i] = min(1.0, run)
    return adj

In [ ]:
# ---------- 5. E1/E2: held-out evaluation (80/20) on D1 and D2, both tasks ----------
HOLDOUT = {}; rows = []
for dname, df in [("D1", d1), ("D2", d2)]:
    for task in ["binary", "subtype"]:
        X, y = task_data(df, task)
        tr, te, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
        preds = {}
        for mname, mk in model_zoo(task).items():
            t0 = time.time(); m = mk().fit(tr, ytr); p = m.predict(te); preds[mname] = p
            P, Rr, F, _ = precision_recall_fscore_support(yte, p, average="macro", zero_division=0)
            row = {"Dataset": dname, "Task": task, "Model": mname, "n_test": len(yte), "Accuracy": accuracy_score(yte, p),
                   "Macro-P": P, "Macro-R": Rr, "Macro-F1": F, "Weighted-F1": f1_score(yte, p, average="weighted"),
                   "Macro-F1 95% CI": boot_ci(lambda i: f1_score(yte[i], p[i], average="macro"), len(yte)), "Train+test time (s)": time.time() - t0}
            if task == "binary":
                pr = m.proba(te)
                if pr is not None: row["Brier"] = brier_score_loss(yte == "NFR", pr); row["AP (NFR)"] = average_precision_score(yte == "NFR", pr)
            if mname == PROPOSED:
                HOLDOUT[(dname, task)] = {"y": yte, "p": p, "proba": m.proba(te) if task == "binary" else None, "labels": sorted(set(y)), "model": m}
                if task == "binary":
                    Pc, Rc, Fc, Sc = precision_recall_fscore_support(yte, p, labels=["FR", "NFR"], zero_division=0)
                    row["per_class"] = {c: [round(Pc[i], 4), round(Rc[i], 4), round(Fc[i], 4), int(Sc[i])] for i, c in enumerate(["FR", "NFR"])}
                    row["confusion[FR,NFR]"] = confusion_matrix(yte, p, labels=["FR", "NFR"]).tolist()
                    if dname == "D1": row["best_C"] = m.best
            rows.append(row); print(dname, task, mname, round(F, 4))
        for mname, p in preds.items():
            if mname != PROPOSED: rows.append({"Dataset": dname, "Task": task, "Model": f"McNemar p: proposed vs {mname}", "p": mcnemar(yte, preds[PROPOSED], p)})
        # per-label table for the proposed model
        h = HOLDOUT[(dname, task)]
        if task == "subtype":
            labs = sorted(set(y)); P_, R_, F_, S_ = precision_recall_fscore_support(h["y"], h["p"], labels=labs, zero_division=0)
            save_tab(pd.DataFrame({"Subtype": labs, "Support": S_, "Precision": P_, "Recall": R_, "F1": F_,
                                   "F1 95% CI": [boot_ci(lambda i, l=l: f1_score(h["y"][i] == l, h["p"][i] == l, zero_division=0), len(h["y"])) for l in labs]}
                                  ).sort_values("Support", ascending=False).round(4), f"table_subtypes_{dname}")
hold = pd.DataFrame(rows); R["holdout"] = hold.to_dict(orient="records")
save_tab(hold[hold.Model.str.startswith("McNemar") == False].drop(columns=[c for c in ["p"] if c in hold]).round(4), "table_holdout_all_models")
save_tab(hold[hold.Model.str.startswith("McNemar")][["Dataset", "Task", "Model", "p"]].round(4), "table_holdout_mcnemar")

In [ ]:
# ---------- Figures: confusion matrices, reliability (proposed, binary) ----------
fig, axes = plt.subplots(1, 2, figsize=(6.4, 2.8))
for ax, d in zip(axes, ["D1", "D2"]):
    h = HOLDOUT[(d, "binary")]; cm = confusion_matrix(h["y"], h["p"], labels=["FR", "NFR"]); ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm): ax.text(j, i, v, ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], ["FR", "NFR"]); ax.set_yticks([0, 1], ["FR", "NFR"]); ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(d)
save_fig("fig_confusion_D1_D2")
fig, ax = plt.subplots(figsize=(3.6, 2.8)); ax.plot([0, 1], [0, 1], "--", c="grey", label="Perfect calibration")
for d in ["D1", "D2"]:
    h = HOLDOUT[(d, "binary")]; fr, mp = calibration_curve(h["y"] == "NFR", h["proba"], n_bins=10)
    ax.plot(mp, fr, "o-", ms=3, label=f"{d} (Brier = {brier_score_loss(h['y'] == 'NFR', h['proba']):.3f})")
ax.set_xlabel("Mean predicted probability (NFR)"); ax.set_ylabel("Observed fraction NFR"); ax.legend(); ax.grid(alpha=.3)
save_fig("fig_reliability_D1_D2")

In [ ]:
# ---------- 6. E3: repeated 5x2 CV with significance tests ----------
CV = defaultdict(list)    # (dataset, task, model) -> list of 10 macro-F1
for dname, df in [("D1", d1), ("D2", d2)]:
    for task in ["binary", "subtype"]:
        X, y = task_data(df, task)
        rskf = RepeatedStratifiedKFold(n_splits=2, n_repeats=5, random_state=SEED)
        for fold, (a, b) in enumerate(rskf.split(X, y)):
            for mname, mk in model_zoo(task, for_cv=True).items():
                m = mk().fit(X.iloc[a], y[a]); CV[(dname, task, mname)].append(f1_score(y[b], m.predict(X.iloc[b]), average="macro"))
            print(dname, task, "fold", fold + 1, "done")
rows = []
for (dname, task) in itertools.product(["D1", "D2"], ["binary", "subtype"]):
    base_ = np.array(CV[(dname, task, PROPOSED)]); others = [k[2] for k in CV if k[:2] == (dname, task) and k[2] != PROPOSED]
    pw = [stats.wilcoxon(base_, np.array(CV[(dname, task, o)])).pvalue if np.any(base_ != np.array(CV[(dname, task, o)])) else 1.0 for o in others]
    pa = [alpaydin_f((base_ - np.array(CV[(dname, task, o)])).reshape(5, 2)) for o in others]
    rows.append({"Dataset": dname, "Task": task, "Model": PROPOSED, "Mean macro-F1": base_.mean(), "SD": base_.std(ddof=1)})
    for o, p1, p1h, p2 in zip(others, pw, holm(pw), pa):
        v = np.array(CV[(dname, task, o)])
        rows.append({"Dataset": dname, "Task": task, "Model": o, "Mean macro-F1": v.mean(), "SD": v.std(ddof=1),
                     "Wilcoxon p": p1, "Wilcoxon p (Holm)": p1h, "5x2cv F-test p": p2, "A12 (proposed vs model)": a12(base_, v)})
cvtab = pd.DataFrame(rows).round(4); save_tab(cvtab, "table_cv5x2_significance")
R["cv5x2"] = {"|".join(k): v for k, v in CV.items()}
fig, axes = plt.subplots(1, 4, figsize=(11, 3), sharey=False)
for ax, (dname, task) in zip(axes, itertools.product(["D1", "D2"], ["binary", "subtype"])):
    ks = [k for k in CV if k[:2] == (dname, task)]; ax.boxplot([CV[k] for k in ks], vert=False)
    ax.set_yticks(range(1, len(ks) + 1), [k[2].replace("TF-IDF + ", "") for k in ks], fontsize=7); ax.set_title(f"{dname} – {task}", fontsize=8)
    ax.set_xlabel("Macro-F1 (5×2 CV)")
save_fig("fig_cv_boxplots")

In [ ]:
# ---------- 7. E4: cross-project (leave-one-project-out) on PROMISE_exp ----------
rows = []
lopo_models = {PROPOSED: lambda: TfidfModel("linsvc"), "TF-IDF + Logistic Regression": lambda: TfidfModel("lr")}
if RUN_NEURAL: lopo_models["SBERT + Logistic Regression"] = lambda: SbertLR()
for task in ["binary", "subtype"]:
    X, y = task_data(d2, task)
    for proj in sorted(X.project.unique()):
        te = (X.project == proj).values
        if te.sum() < 5 or len(set(y[~te])) < 2: continue
        for mname, mk in lopo_models.items():
            m = mk().fit(X[~te], y[~te]); p = m.predict(X[te])
            rows.append({"Task": task, "Project": proj, "n": int(te.sum()), "Model": mname,
                         "Macro-F1": f1_score(y[te], p, average="macro"), "Weighted-F1": f1_score(y[te], p, average="weighted"), "Accuracy": accuracy_score(y[te], p)})
lopo = pd.DataFrame(rows); save_tab(lopo.round(4), "table_lopo_promise_by_project")
lopo_sum = lopo.groupby(["Task", "Model"]).agg(**{"Mean weighted-F1": ("Weighted-F1", "mean"), "SD": ("Weighted-F1", "std"),
                                                 "Mean macro-F1": ("Macro-F1", "mean"), "Mean accuracy": ("Accuracy", "mean")}).reset_index()
save_tab(lopo_sum.round(4), "table_lopo_promise_summary"); R["lopo_summary"] = lopo_sum.to_dict(orient="records")

In [ ]:
# ---------- 8. E5: cross-dataset transfer (synthetic D1 <-> real D2), FR/NFR ----------
rows = []
transfer_models = {PROPOSED: lambda: TfidfModel("linsvc")}
if RUN_NEURAL: transfer_models["SBERT + Logistic Regression"] = lambda: SbertLR()
if RUN_BERT and RUN_NEURAL: transfer_models["Fine-tuned BERT"] = lambda: BertClf()
for (sn, s), (tn, t) in [(("D1", d1), ("D2", d2)), (("D2", d2), ("D1", d1))]:
    for mname, mk in transfer_models.items():
        m = mk().fit(s, s.type.values); p = m.predict(t)
        rows.append({"Train": sn, "Test": tn, "Model": mname, "Accuracy": accuracy_score(t.type, p), "Macro-F1": f1_score(t.type, p, average="macro"),
                     "Macro-F1 95% CI": boot_ci(lambda i: f1_score(t.type.values[i], p[i], average="macro"), len(t))})
transfer = pd.DataFrame(rows); save_tab(transfer.round(4), "table_cross_dataset"); R["cross_dataset"] = transfer.to_dict(orient="records")

In [ ]:
# ---------- 9. E6: ablations of the proposed pipeline (5x2 CV, macro-F1) ----------
ABL = {"Full pipeline": dict(),
       "– lemmatisation": dict(text_col="clean_nolem"),
       "– stop-word removal": dict(text_col="clean_nostop"),
       "Unigrams only": dict(ngram=(1, 1)),
       "– oversampling": dict(oversample=False)}
rows = []
for dname, df in [("D1", d1), ("D2", d2)]:
    for task in ["binary", "subtype"]:
        X, y = task_data(df, task); scores = defaultdict(list)
        for a, b in RepeatedStratifiedKFold(n_splits=2, n_repeats=5, random_state=SEED).split(X, y):
            for an, kw in ABL.items():
                m = TfidfModel("linsvc", calibrate=None, **kw).fit(X.iloc[a], y[a]); scores[an].append(f1_score(y[b], m.predict(X.iloc[b]), average="macro"))
        full = np.array(scores["Full pipeline"])
        for an in ABL:
            v = np.array(scores[an]); p = stats.wilcoxon(full, v).pvalue if an != "Full pipeline" and np.any(full != v) else np.nan
            rows.append({"Dataset": dname, "Task": task, "Variant": an, "Mean macro-F1": v.mean(), "SD": v.std(ddof=1), "Δ vs full": v.mean() - full.mean(), "Wilcoxon p": p})
abl = pd.DataFrame(rows); save_tab(abl.round(4), "table_ablation"); R["ablation"] = abl.to_dict(orient="records")
# calibration ablation on D1/D2 hold-out
rows = []
for dname, df in [("D1", d1), ("D2", d2)]:
    tr, te = train_test_split(df, test_size=0.2, stratify=df.type, random_state=SEED)
    for meth in ["isotonic", "sigmoid"]:
        m = TfidfModel("linsvc", calibrate=meth).fit(tr, tr.type.values); pr = m.proba(te)
        rows.append({"Dataset": dname, "Calibration": meth, "Brier": brier_score_loss(te.type == "NFR", pr),
                     "Macro-F1": f1_score(te.type, m.predict(te), average="macro")})
save_tab(pd.DataFrame(rows).round(4), "table_calibration")

In [ ]:
# ---------- 10. E7: FR→NFR linking — lexical vs semantic vs hybrid ----------
from rank_bm25 import BM25Okapi
def sim_matrices(fr_texts, nfr_texts, fit_texts):
    fr_c = [preprocess(t) for t in fr_texts]; nf_c = [preprocess(t) for t in nfr_texts]
    vec = make_vec().fit([preprocess(t) for t in fit_texts])
    S = {"TF-IDF cosine": (vec.transform(fr_c) @ vec.transform(nf_c).T).toarray()}
    bm = BM25Okapi([d.split() or ["_"] for d in nf_c]); B = np.array([bm.get_scores(q.split()) for q in fr_c])
    S["BM25"] = B / (B.max(1, keepdims=True) + 1e-9)
    if RUN_NEURAL: S["SBERT cosine"] = embed(list(fr_texts)) @ embed(list(nfr_texts)).T
    return S
def rank_metrics(S, G, ks=(1, 3, 5)):
    out = defaultdict(list)
    for q in range(S.shape[0]):
        rel = G[q]
        if rel.sum() == 0: continue
        hits = rel[np.argsort(-S[q], kind="stable")]
        for k in ks: out[f"P@{k}"].append(hits[:k].mean())
        cum = np.cumsum(hits); pos = np.where(hits)[0]; out["AP"].append(np.mean(cum[pos] / (pos + 1))); out["RR"].append(1 / (pos[0] + 1))
    return out
def pairwise(S, G, tau):
    acc = S >= tau; tp = (acc & G).sum(); P_ = tp / max(acc.sum(), 1); R_ = tp / max(G.sum(), 1)
    return P_, R_, (2 * P_ * R_ / (P_ + R_) if P_ + R_ else 0.0), float((acc.sum(1) > 0).mean())
TAUS = np.round(np.arange(0.0, 0.951, 0.025), 3); ALPHAS = np.round(np.arange(0, 1.01, 0.1), 1)
def combine(S, alpha): return alpha * S["TF-IDF cosine"] + (1 - alpha) * S["SBERT cosine"]

# ---- D3: PROMISE gold traces, leave-one-project-out tuning of alpha and tau ----
proj_S = {}
all_promise_text = promise_nfr.text.tolist()
for p, v in D3.items():
    others = promise_nfr[promise_nfr.project != p].text.tolist()       # IDF fitted without the target project
    proj_S[p] = sim_matrices(v["fr"], v["nfr"], others)
methods = list(proj_S[1].keys()) + (["Hybrid (TF-IDF + SBERT)"] if RUN_NEURAL else [])
def scores_for(p, method, alpha=None): return combine(proj_S[p], alpha) if method.startswith("Hybrid") else proj_S[p][method]
rows, perq = [], defaultdict(list)
for p in D3:
    G = D3[p]["G"]; train_ps = [q for q in D3 if q != p]
    for method in methods:
        alpha = None
        if method.startswith("Hybrid"):        # alpha chosen by MAP on the other 9 projects
            alpha = max(ALPHAS, key=lambda a: np.mean([np.mean(rank_metrics(scores_for(q, method, a), D3[q]["G"])["AP"]) for q in train_ps]))
        tau = max(TAUS, key=lambda t: np.mean([pairwise(scores_for(q, method, alpha), D3[q]["G"], t)[2] for q in train_ps]))
        S = scores_for(p, method, alpha); rm = rank_metrics(S, G); pP, pR, pF, cov = pairwise(S, G, tau)
        perq[method] += rm["AP"]
        rows.append({"Project": p, "Method": method, "alpha": alpha, "tau": tau, "MAP": np.mean(rm["AP"]), "MRR": np.mean(rm["RR"]),
                     "P@1": np.mean(rm["P@1"]), "P@3": np.mean(rm["P@3"]), "Pairwise P": pP, "Pairwise R": pR, "Pairwise F1": pF, "Coverage": cov})
    # random baseline (expected values over 200 shuffles)
    rng = np.random.default_rng(SEED); rr = defaultdict(list)
    for _ in range(200):
        Sr = rng.random(G.shape); m_ = rank_metrics(Sr, G)
        for k in ["AP", "RR", "P@1", "P@3"]: rr[k].append(np.mean(m_[k]))
    rows.append({"Project": p, "Method": "Random ranking", "MAP": np.mean(rr["AP"]), "MRR": np.mean(rr["RR"]), "P@1": np.mean(rr["P@1"]), "P@3": np.mean(rr["P@3"])})
    d_ = float(G.mean())   # accepting every FR-NFR pair: precision = link density, recall = 1
    rows.append({"Project": p, "Method": "Accept-all pairs", "Pairwise P": d_, "Pairwise R": 1.0, "Pairwise F1": 2 * d_ / (1 + d_), "Coverage": 1.0})
link_d3 = pd.DataFrame(rows); save_tab(link_d3.round(4), "table_linking_D3_by_project")
link_d3_sum = link_d3.groupby("Method")[["MAP", "MRR", "P@1", "P@3", "Pairwise P", "Pairwise R", "Pairwise F1", "Coverage"]].mean().reset_index()
sig = []
for mth in methods:
    if mth != "TF-IDF cosine":
        a_, b_ = np.array(perq[mth]), np.array(perq["TF-IDF cosine"])
        sig.append({"Comparison": f"{mth} vs TF-IDF cosine (per-query AP)", "n queries": len(a_), "Wilcoxon p": stats.wilcoxon(a_, b_).pvalue if np.any(a_ != b_) else 1.0, "A12": a12(a_, b_)})
save_tab(link_d3_sum.round(4), "table_linking_D3_summary"); save_tab(pd.DataFrame(sig).round(4), "table_linking_D3_significance")
R["linking_D3"] = link_d3_sum.to_dict(orient="records"); R["linking_D3_sig"] = sig

# similarity distributions: linked vs non-linked pairs (pooled D3)
fig, axes = plt.subplots(1, len(methods) - (1 if RUN_NEURAL else 0), figsize=(3.2 * (len(methods) - (1 if RUN_NEURAL else 0)), 2.5))
for ax, mth in zip(np.atleast_1d(axes), [m for m in methods if not m.startswith("Hybrid")]):
    pos = np.concatenate([proj_S[p][mth][D3[p]["G"]] for p in D3]); neg = np.concatenate([proj_S[p][mth][~D3[p]["G"]] for p in D3])
    ax.hist(neg, bins=40, alpha=.6, density=True, label="Not linked"); ax.hist(pos, bins=40, alpha=.6, density=True, label="Linked (gold)")
    ax.set_title(mth, fontsize=8); ax.set_xlabel("Similarity"); ax.legend(fontsize=7)
save_fig("fig_linking_similarity_distributions_D3")

# ---- D1: silver labels (shared module), with and without the cross-cutting 'System Wide' module ----
tr1, te1 = train_test_split(d1, test_size=0.2, stratify=d1.type + "|" + d1["sub"], random_state=SEED)
fit1, dev1 = train_test_split(tr1, test_size=0.2, stratify=tr1.type + "|" + tr1["sub"], random_state=SEED)
def d1_link(part, fit_texts):
    fr = part[part.type == "FR"]; nf = part[part.type == "NFR"]
    S = sim_matrices(fr.text.tolist(), nf.text.tolist(), fit_texts)
    a = fr.module.to_numpy(dtype=object); b = nf.module.to_numpy(dtype=object)
    G_strict = a[:, None] == b[None, :]
    G_sw = G_strict | (b[None, :] == "System Wide")       # variant: system-wide NFRs constrain every FR
    return S, {"Silver: same module": G_strict, "Silver: same module + System Wide": G_sw}
Sdev, Gdev = d1_link(dev1, fit1.text.tolist()); Ste, Gte = d1_link(te1, tr1.text.tolist())
rows = []
for gname in Gte:
    for method in methods:
        alpha = None
        if method.startswith("Hybrid"):
            alpha = max(ALPHAS, key=lambda a: np.mean(rank_metrics(combine(Sdev, a), Gdev[gname])["AP"]))
        Sd = combine(Sdev, alpha) if alpha is not None else Sdev[method]; St = combine(Ste, alpha) if alpha is not None else Ste[method]
        tau = max(TAUS, key=lambda t: pairwise(Sd, Gdev[gname], t)[2])
        rm = rank_metrics(St, Gte[gname]); pP, pR, pF, cov = pairwise(St, Gte[gname], tau)
        rows.append({"Silver labels": gname, "Method": method, "alpha": alpha, "tau": tau, "Queries with ≥1 relevant": len(rm["AP"]),
                     "MAP": np.mean(rm["AP"]), "MRR": np.mean(rm["RR"]), "P@1": np.mean(rm["P@1"]), "P@3": np.mean(rm["P@3"]),
                     "Pairwise P": pP, "Pairwise R": pR, "Pairwise F1": pF, "Coverage": cov,
                     "Random P@1": float(Gte[gname][Gte[gname].sum(1) > 0].mean())})
link_d1 = pd.DataFrame(rows); save_tab(link_d1.round(4), "table_linking_D1_silver"); R["linking_D1"] = link_d1.to_dict(orient="records")
R["D1_system_wide_share"] = float((d1.module == "System Wide").mean())

fig, ax = plt.subplots(figsize=(4.6, 2.6)); w = 0.8 / max(1, len(methods))
for i, mth in enumerate(methods + ["Random ranking"]):
    v = link_d3_sum.set_index("Method").reindex([mth])["MAP"].values[0]; ax.bar(i, v); ax.text(i, v + .01, f"{v:.2f}", ha="center", fontsize=7)
ax.set_xticks(range(len(methods) + 1), [m.replace(" (TF-IDF + SBERT)", "") for m in methods] + ["Random"], rotation=20, fontsize=7)
ax.set_ylabel("MAP (PROMISE gold traces)")
save_fig("fig_linking_map_D3")

In [ ]:
# ---------- 11. E8: efficiency (inference latency per 1,000 statements) ----------
rows = []; sample = d1.sample(min(1000, len(d1)), random_state=SEED)
def lat(f, reps=3):
    ts = []
    for _ in range(reps): t = time.perf_counter(); f(); ts.append(time.perf_counter() - t)
    return float(np.median(ts))
m = TfidfModel("linsvc").fit(d1, d1.type.values); rows.append({"Model": PROPOSED, "Seconds per 1,000": lat(lambda: m.predict(sample)) * 1000 / len(sample)})
if RUN_NEURAL:
    ms = SbertLR().fit(d1, d1.type.values)
    rows.append({"Model": "SBERT + Logistic Regression", "Seconds per 1,000": lat(lambda: SBERT.encode(sample.text.tolist(), batch_size=64)) * 1000 / len(sample)})
if RUN_BERT and RUN_NEURAL:
    mb = BertClf().fit(d1, d1.type.values); rows.append({"Model": "Fine-tuned BERT", "Seconds per 1,000": lat(lambda: mb.predict(sample)) * 1000 / len(sample)})
eff = pd.DataFrame(rows); eff["Device"] = R["env"].get("device", "cpu"); eff["RSS (MB)"] = psutil.Process().memory_info().rss / 2**20
save_tab(eff.round(4), "table_efficiency"); R["efficiency"] = eff.to_dict(orient="records")

In [ ]:
# ---------- 12. Interpretability: top n-grams and errors (proposed, D1 binary) ----------
h = HOLDOUT[("D1", "binary")]; mm = h["model"]; lin = mm.model.named_steps["m"]; fn = np.array(mm.vec.get_feature_names_out()); w_ = lin.coef_[0]
pos_cls = lin.classes_[1]
R["top_ngrams_D1"] = {pos_cls: fn[np.argsort(-w_)[:15]].tolist(), lin.classes_[0]: fn[np.argsort(w_)[:15]].tolist()}
print(R["top_ngrams_D1"])

In [ ]:
# ---------- 13. Save ----------
def to_jsonable(o):
    if isinstance(o, dict): return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_jsonable(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return o
json.dump(to_jsonable(R), open(f"{OUT}/results_v2.json", "w"), indent=1, default=str)
with zipfile.ZipFile("results_v2_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT): z.write(f"{OUT}/{f}", f"results_v2/{f}")
print("Saved results_v2_bundle.zip")
try:
    from google.colab import files; files.download("results_v2_bundle.zip")
except ImportError: pass